# Customer Churn Prediction

I built this notebook to understand which customers are more likely to leave a telecom service.  
The workflow is kept simple: inspect the data, clean it, prepare the categorical columns, train two classifiers, and compare their results.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

df = pd.read_csv("Customer_Churn_Analysis_Custom.csv")
df.head()

## 1. Quick look at the data

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nMissing values:")
print(df.isna().sum())

print("\nChurn counts:")
print(df["Churn"].value_counts())

In [ ]:
# CustomerID is only an identifier, so it is not useful for prediction.
df = df.drop(columns=["CustomerID"])

# Convert the target into 0/1.
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

# TotalCharges can have a few missing values; the pipeline will handle them.
df.describe(include="all").T

## 2. A couple of simple checks

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
df["Churn"].value_counts().sort_index().plot(kind="bar", ax=ax)
ax.set_xticklabels(["Stayed", "Churned"], rotation=0)
ax.set_ylabel("Customers")
ax.set_title("Customer Churn Distribution")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
df.groupby("Contract")["Churn"].mean().sort_values().plot(kind="bar", ax=ax)
ax.set_ylabel("Churn rate")
ax.set_title("Average churn rate by contract")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 3. Prepare the features

In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

numeric_features = ["Age", "Tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = ["Gender", "PhoneService", "InternetService", "Contract"]

numeric_pipe = Pipeline([
    ("fill", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

categorical_pipe = Pipeline([
    ("fill", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 4. Train Logistic Regression

In [ ]:
logistic_model = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

logistic_model.fit(X_train, y_train)
logistic_pred = logistic_model.predict(X_test)
logistic_prob = logistic_model.predict_proba(X_test)[:, 1]

print("Logistic Regression accuracy:", round(accuracy_score(y_test, logistic_pred), 3))
print("Logistic Regression ROC-AUC:", round(roc_auc_score(y_test, logistic_prob), 3))
print("\nClassification report:")
print(classification_report(y_test, logistic_pred))

## 5. Try Random Forest

In [ ]:
forest_model = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=250,
        max_depth=8,
        min_samples_leaf=3,
        random_state=42,
        class_weight="balanced"
    ))
])

forest_model.fit(X_train, y_train)
forest_pred = forest_model.predict(X_test)
forest_prob = forest_model.predict_proba(X_test)[:, 1]

print("Random Forest accuracy:", round(accuracy_score(y_test, forest_pred), 3))
print("Random Forest ROC-AUC:", round(roc_auc_score(y_test, forest_prob), 3))
print("\nClassification report:")
print(classification_report(y_test, forest_pred))

## 6. Compare the models

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, logistic_pred),
        accuracy_score(y_test, forest_pred)
    ],
    "ROC_AUC": [
        roc_auc_score(y_test, logistic_prob),
        roc_auc_score(y_test, forest_prob)
    ]
}).round(3)

results

In [ ]:
cm = confusion_matrix(y_test, forest_pred)

fig, ax = plt.subplots(figsize=(5,4))
ax.imshow(cm)
ax.set_xticks([0,1], labels=["Stayed","Churned"])
ax.set_yticks([0,1], labels=["Stayed","Churned"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Random Forest Confusion Matrix")

for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()

## 7. Try one customer

The model can also return a probability instead of only Yes/No. This is useful because a customer with a 0.51 probability is different from one with a 0.95 probability.

In [ ]:
sample_customer = X_test.iloc[[0]]
prediction = forest_model.predict(sample_customer)[0]
probability = forest_model.predict_proba(sample_customer)[0, 1]

print("Predicted churn:", "Yes" if prediction == 1 else "No")
print("Churn probability:", round(probability, 3))

## Conclusion

The project compares a simple linear classifier with a tree-based model.  
ROC-AUC is included along with accuracy because churn datasets can have more customers who stay than customers who leave.

The main takeaway is not just the final score, but the complete flow from raw customer records to a model that can estimate churn risk.